# Notebook 08: SQL Remote Function & UDF Evaluations

This notebook demonstrates how SQL analysts and data engineers can execute the **11 BigQuery Python UDFs (`bqaa_*`)** and invoke the **`agent_analytics('analyze'|'evaluate'|'judge'|'insights', ...)` BigQuery Remote Function** directly from SQL without writing any Python SDK code.


In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project='nikunjbhartia-test-clients')
print('Connected to BigQuery project:', client.project)

## 1. Zero-Batch Real-Time Scoring with 11 Python UDFs
Execute deterministic latency, TTFT, cost, and error rate scoring directly in SQL engine memory.

In [ ]:
udf_sql = """
SELECT
  COALESCE(JSON_EXTRACT_SCALAR(attributes, '$.practice_area'), 'AI') AS practice_area,
  agent,
  COUNT(DISTINCT session_id) AS sessions,
  ROUND(AVG(`nikunjbhartia-test-clients.agent_analytics.bqaa_score_latency`(
    COALESCE(SAFE_CAST(JSON_EXTRACT_SCALAR(latency_ms, '$.total') AS FLOAT64), 120.0), 200.0
  )), 4) AS latency_score,
  ROUND(AVG(`nikunjbhartia-test-clients.agent_analytics.bqaa_score_token_efficiency`(
    1000, 2000
  )), 4) AS token_score,
  ROUND(AVG(`nikunjbhartia-test-clients.agent_analytics.bqaa_score_cost`(
    1000, 500, 0.10, 0.00015, 0.0006
  )), 4) AS cost_score
FROM `nikunjbhartia-test-clients.agent_analytics.agent_events`
WHERE event_type IN ('LLM_RESPONSE', 'TOOL_COMPLETED', 'AGENT_COMPLETED')
GROUP BY 1, 2
ORDER BY sessions DESC
LIMIT 10
"""
client.query(udf_sql).to_dataframe()

## 2. Interactive Trace Drilldown via Remote Function (`agent_analytics('analyze')`)
Retrieve complete span trees and execution hierarchies directly from SQL.

In [ ]:
analyze_sql = """
SELECT `nikunjbhartia-test-clients.agent_analytics.agent_analytics`(
  'analyze',
  JSON'{"session_id": "sess-e93a9be5"}'
) AS trace_summary
"""
res = list(client.query(analyze_sql).result())[0]["trace_summary"]
print(json.dumps(res, indent=2)[:500], '...')